In [1]:
import numpy as np
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
from gensim.models import Word2Vec

In [3]:
import torch
import math

# **self-attention class**

In [4]:
class SelfAttention:

    # to initialize it
    def __init__(self, dimen=4, heads=1, random_w=True):

        self.dimen = dimen
        self.random_w = random_w
        self.heads = heads

        if random_w:
            self.w_query = torch.rand(dimen, dimen//heads)
            self.w_key = torch.rand(dimen, dimen//heads)
            self.w_value = torch.rand(dimen, dimen//heads)
        else:
            self.w_query = None
            self.w_key = None
            self.w_value = None

    def getQKV(self, static_embeddings):

        if self.random_w == False:
            
            queries = static_embeddings
            keys = static_embeddings
            values = static_embeddings
            
        else:
        
            queries = torch.matmul(static_embeddings, self.w_query)
            keys = torch.matmul(static_embeddings, self.w_key)
            values = torch.matmul(static_embeddings, self.w_value)

        return queries, keys, values

    def dotproduct(self, queries, keys):

        dot_product = torch.matmul(queries,keys.T)
        return dot_product

    def get_sqrt(self, dot_product):

        sqroot = math.sqrt(self.dimen)
        return (1/sqroot)*dot_product

    def apply_softmax(self, scaled_dp):
        
        weights = torch.softmax(scaled_dp.T, dim=0).T
        return weights

    def mul_weights_and_values(self, weights, values):

        contextual_embeddings = torch.matmul(weights, values)

        #print(contextual_embeddings)
        return contextual_embeddings
        

    # get embeddings of different sentences
    def __call__(self, static_embeddings):

        # 1. obtain query, key, value vectors
        queries, keys, values = self.getQKV(static_embeddings)

        # 2. perform dot product bw query and key
        dot_product = self.dotproduct(queries,keys)

        # 3. scale by 1/sqrt(dimen)
        scaled_dp = self.get_sqrt(dot_product)

        # 4. apply softmax to each row
        weights = self.apply_softmax(scaled_dp)

        # 5. multiply w value to get final embeddings
        contextual_embeddings = self.mul_weights_and_values(weights,values)
        return contextual_embeddings

In [5]:
sentence1 = "River bank flows"
s1 = sentence1.split(" ")
print(s1)

['River', 'bank', 'flows']


In [6]:
sentence2 = "Money bank grows"
s2 = sentence2.split(" ")
print(s2)

['Money', 'bank', 'grows']


In [7]:
sentences = [s1, s2]
print(sentences)

[['River', 'bank', 'flows'], ['Money', 'bank', 'grows']]


In [8]:
model = Word2Vec(sentences, vector_size=256, window=5, min_count=1, sg=1)

In [9]:
input_matrix1 = []

for word in s1:
    embedding = model.wv[word]
    input_matrix1.append(embedding)

np_ip1 = np.array(input_matrix1)
static_embeddings1 = torch.tensor(np_ip1)

#print(query_tensors)

In [10]:
sa_block = SelfAttention(256,True)

In [11]:
ce1 = sa_block(static_embeddings1)

In [12]:
input_matrix2 = []

for word in s2:
    embedding = model.wv[word]
    input_matrix2.append(embedding)

np_ip2 = np.array(input_matrix2)
static_embeddings2 = torch.tensor(np_ip2)

In [13]:
ce2 = sa_block(static_embeddings2)

In [14]:
print(static_embeddings2[0:2])

tensor([[-2.8321e-03, -3.7513e-03, -1.0717e-03, -3.2667e-03, -2.3589e-03,
         -2.2152e-03, -9.1568e-04, -6.6680e-04, -3.4988e-03, -2.8719e-04,
          3.1846e-03,  3.0041e-03, -2.8149e-03, -1.4324e-03,  1.2182e-03,
         -3.7386e-03,  5.7673e-04,  2.5486e-03,  2.2447e-03, -3.4231e-03,
         -1.7645e-03, -3.1798e-03,  1.7952e-05,  3.6186e-03,  2.3333e-03,
          1.9794e-03,  1.9770e-03, -1.2668e-03,  3.7313e-03, -2.8736e-03,
         -2.8400e-03, -8.8492e-04, -3.0413e-04, -1.2563e-03, -2.3148e-04,
          2.9253e-03, -2.7247e-04, -6.3474e-04,  1.0720e-03, -3.2653e-03,
          3.0687e-03,  3.3344e-03, -3.7438e-03,  9.5557e-04,  3.8691e-03,
         -2.9945e-03, -2.7215e-03, -3.0221e-03,  3.2797e-03, -2.6615e-04,
          3.5720e-03, -3.1868e-03,  1.4621e-03,  1.0293e-03,  2.9012e-04,
          9.0925e-04, -2.9176e-03, -3.6556e-03,  9.1976e-04,  2.4017e-03,
          3.1194e-03,  2.2406e-03, -3.0365e-04,  3.2446e-03, -3.6470e-03,
          1.3305e-03,  1.0420e-04,  1.

# **multi-head attention**

In [15]:
ambiguous_sentence = "The cat chased the mouse until it stumbled"
ambiguous_sentence = ambiguous_sentence.split(" ")
print(ambiguous_sentence)

['The', 'cat', 'chased', 'the', 'mouse', 'until', 'it', 'stumbled']


In [16]:
def get_embeddings(model, sentence):
    
    embeddings = [model.wv[word] for word in sentence]
    embeddings_np = np.array(embeddings)
    return torch.tensor(embeddings_np)

In [17]:
model = Word2Vec([ambiguous_sentence], vector_size=512, window=5, min_count=1, sg=1)

In [18]:
emb = get_embeddings(model,ambiguous_sentence)
print(emb.shape)

torch.Size([8, 512])


In [19]:
print(emb)

tensor([[-8.3858e-04, -1.2931e-03, -2.7027e-04,  ..., -1.9349e-03,
         -2.9324e-04,  1.3476e-03],
        [ 1.2903e-03, -2.2377e-04,  1.4973e-03,  ...,  1.0317e-04,
         -1.7566e-03,  1.6347e-03],
        [ 1.1490e-03, -5.7839e-04,  6.1753e-04,  ..., -1.6250e-03,
         -2.8566e-05, -5.1714e-04],
        ...,
        [-1.2890e-03,  8.4774e-04, -9.2673e-05,  ...,  1.8470e-03,
         -1.1359e-03,  1.6143e-03],
        [-1.4161e-03, -1.8756e-03, -5.3587e-04,  ..., -5.0609e-04,
          1.4158e-03, -6.7645e-04],
        [-1.0473e-04,  4.6178e-05,  9.9675e-04,  ..., -1.3424e-03,
         -9.7645e-04, -4.4665e-04]])


In [20]:
sa1 = SelfAttention(512,8,True)
sa2 = SelfAttention(512,8,True)
sa3 = SelfAttention(512,8,True)
sa4 = SelfAttention(512,8,True)
sa5 = SelfAttention(512,8,True)
sa6 = SelfAttention(512,8,True)
sa7 = SelfAttention(512,8,True)
sa8 = SelfAttention(512,8,True)

In [21]:
e1 = sa1(emb)
e2 = sa2(emb)
e3 = sa3(emb)
e4 = sa4(emb)
e5 = sa5(emb)
e6 = sa6(emb)
e7 = sa7(emb)
e8 = sa8(emb)

In [22]:
print(e1.shape)

torch.Size([8, 64])


In [23]:
print(e2)

tensor([[ 3.1680e-05, -4.1791e-03,  3.5995e-03,  6.4295e-03,  4.0933e-03,
          1.6204e-03,  3.2037e-03,  6.5857e-03,  1.1200e-02,  4.0887e-03,
          3.7248e-03,  3.7785e-04,  8.2639e-03,  5.5574e-03,  8.9585e-03,
          6.8688e-03,  8.4725e-03,  5.3205e-03,  2.7501e-03,  5.6827e-03,
          5.4138e-03, -4.2348e-04,  3.9370e-04,  4.0318e-03,  6.2695e-03,
          4.9985e-03,  1.3617e-03,  6.1820e-03,  9.4808e-04,  4.7865e-03,
          3.8408e-03,  4.2282e-03,  4.0429e-03,  3.5342e-03,  7.4316e-03,
          5.3554e-03,  5.4434e-03,  3.6559e-03,  3.9019e-03,  5.3104e-03,
          4.2794e-03,  1.4806e-03,  5.6750e-03,  3.2134e-03, -9.5584e-04,
          7.2109e-03,  3.1957e-03,  7.6347e-03,  8.4009e-04,  3.4219e-03,
          3.4378e-03,  2.5775e-03,  3.7741e-03,  3.8382e-03,  9.2045e-03,
          7.0124e-04,  8.0507e-04,  1.2833e-03,  3.5029e-03,  1.4652e-03,
          3.3304e-03,  4.2892e-03,  5.2753e-03,  2.6024e-03],
        [ 3.5993e-05, -4.1739e-03,  3.6035e-03,  6

In [24]:
concatenated = torch.cat([e1,e2,e3,e4,e5,e6,e7,e8], dim=1)
print(concatenated.shape)

torch.Size([8, 512])


In [25]:
print(concatenated)

tensor([[0.0023, 0.0072, 0.0048,  ..., 0.0033, 0.0043, 0.0069],
        [0.0023, 0.0072, 0.0048,  ..., 0.0033, 0.0043, 0.0069],
        [0.0023, 0.0072, 0.0048,  ..., 0.0033, 0.0043, 0.0069],
        ...,
        [0.0023, 0.0072, 0.0048,  ..., 0.0033, 0.0043, 0.0069],
        [0.0023, 0.0072, 0.0048,  ..., 0.0033, 0.0043, 0.0069],
        [0.0023, 0.0072, 0.0048,  ..., 0.0033, 0.0043, 0.0069]])


In [26]:
class MultiHeadAttention:
    
    def __init__(self, num_heads=1, dim=4, random_w=True):
        
        self.num_heads = num_heads
        self.dim = dim
        self.random_w = random_w

    def __call__(self, emb):
        
        outputs = []
        
        for i in range(self.num_heads):
            sa = SelfAttention(self.dim, self.num_heads, self.random_w)
            ce = sa(emb)
            outputs.append(ce)

        concatenated = torch.cat(outputs, dim=1)
        return concatenated

In [27]:
ma = MultiHeadAttention(8,512,True)
embb = ma(emb)
print(embb.shape)

torch.Size([8, 512])


In [28]:
print(embb)

tensor([[0.0065, 0.0036, 0.0073,  ..., 0.0085, 0.0040, 0.0013],
        [0.0065, 0.0036, 0.0074,  ..., 0.0085, 0.0040, 0.0013],
        [0.0065, 0.0036, 0.0073,  ..., 0.0085, 0.0040, 0.0013],
        ...,
        [0.0065, 0.0036, 0.0074,  ..., 0.0085, 0.0040, 0.0013],
        [0.0065, 0.0036, 0.0073,  ..., 0.0085, 0.0040, 0.0013],
        [0.0065, 0.0036, 0.0074,  ..., 0.0085, 0.0040, 0.0013]])
